# Week 10: Transport Networks

This notebook covers:
- Downloading street networks with OSMnx
- Calculating travel-time isochrones
- Analyzing accessibility from a facility

---

## How does this work?

We use **OSMnx** (OpenStreetMap + NetworkX) to:

1. **Download street data** from OpenStreetMap's servers via the internet
2. **Convert it to a graph** (network of nodes and edges)
3. **Analyze accessibility** (walking times, distances)

This requires an internet connection, but no local data files!

---

## Step 0: Set up environment

In [ ]:
# Detect environment and install packages
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    print("Installing packages (takes ~1-2 minutes)...")
    !pip install geopandas osmnx networkx -q
    print("Done!")
else:
    print("Running locally")
    print("Make sure you activated: conda activate intro-gis")

---

## Step 1: Set up folder structure

This notebook uses a standardized folder structure:
- **data/raw/** → Input data (e.g., facility locations you want to analyze)
- **data/processed/** → Output files (e.g., calculated isochrones)

This notebook downloads network data from the internet, so you don't need any files in data/raw/ to get started. However, if you want to analyze your own facility locations, place them in data/raw/.

In [ ]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path("/content/drive/MyDrive/intro-gis/data")
    RAW = BASE / "raw"
    PROCESSED = BASE / "processed"
    RAW.mkdir(parents=True, exist_ok=True)
    PROCESSED.mkdir(parents=True, exist_ok=True)
    print(f"Raw data folder: {RAW}")
    print(f"Processed data folder: {PROCESSED}")
else:
    BASE = Path("../data")
    RAW = BASE / "raw"
    PROCESSED = BASE / "processed"
    RAW.mkdir(parents=True, exist_ok=True)
    PROCESSED.mkdir(parents=True, exist_ok=True)
    print(f"Raw data folder: {RAW.resolve()}")
    print(f"Processed data folder: {PROCESSED.resolve()}")

---

## Step 2: Import libraries

In [ ]:
import geopandas as gpd
import networkx as nx
import osmnx as ox
import matplotlib.pyplot as plt
from shapely.geometry import Point

# Enable caching (speeds up repeated downloads)
ox.settings.use_cache = True

print("Libraries imported!")

---

## Step 3: Download street network

This downloads real street data from OpenStreetMap. Takes 5-15 seconds.

**Parameters:**
- `LATITUDE, LONGITUDE`: Center point (find coordinates by right-clicking in Google Maps)
- `RADIUS`: How far around the point to download (in meters)
- `network_type`: `"walk"`, `"drive"`, or `"bike"`

In [ ]:
# Study area: University of Melbourne
# Change these to any location you want to analyze!
LATITUDE = -37.7983
LONGITUDE = 144.9610
RADIUS = 750  # meters (750m = ~10 min walking radius)

print(f"Downloading walking network: {RADIUS}m around ({LATITUDE}, {LONGITUDE})")
print("This takes about 5-15 seconds...")

G = ox.graph_from_point(
    (LATITUDE, LONGITUDE),
    dist=RADIUS,
    network_type="walk"
)

print(f"\nDownload complete!")
print(f"Nodes (intersections): {len(G.nodes)}")
print(f"Edges (street segments): {len(G.edges)}")

---

## Step 4: Visualize the network

The network is a "graph" with:
- **Nodes** = intersections
- **Edges** = street segments

In [ ]:
fig, ax = ox.plot_graph(G, figsize=(10, 10), node_size=0, edge_linewidth=0.5)
print("Street network visualization")

---

## Step 5: Define facility location

A "facility" is any location you want to measure accessibility FROM (hospital, school, train station, etc.).

You can either:
1. Use the network center (default below)
2. Load your own facilities file from **data/raw/** folder

In [ ]:
# Check if user has a facilities file in data/raw/
facilities_path = RAW / "facilities.geojson"

if facilities_path.exists():
    facilities = gpd.read_file(facilities_path).to_crs(4326)
    print(f"Loaded {len(facilities)} facilities from {facilities_path}")
else:
    # Create a sample facility at the network center
    facilities = gpd.GeoDataFrame(
        {"name": ["Study Area Center"]},
        geometry=[Point(LONGITUDE, LATITUDE)],
        crs="EPSG:4326"
    )
    print("Using network center as sample facility")
    print(f"(Upload facilities.geojson to {RAW}/ to use your own locations)")

facilities

---

## Step 6: Calculate isochrones

**What is an isochrone?**

An isochrone shows all locations reachable within a certain travel time.
- "iso" (equal) + "chronos" (time)
- A 10-minute isochrone = everywhere you can walk in 10 minutes

**How it works:**
1. Find the nearest network node to the facility
2. Calculate walking distance for each time (5 min × 80 m/min = 400m)
3. Find all nodes within that distance
4. Draw a shape around those nodes

In [ ]:
# Walking speed parameters
WALK_SPEED = 4.8  # km/h (average walking speed)
METERS_PER_MIN = WALK_SPEED * 1000 / 60  # = 80 meters per minute

# Time thresholds (minutes)
TIMES = [5, 10, 15]

isochrones = []

for _, fac in facilities.iterrows():
    # Find nearest network node to this facility
    node = ox.distance.nearest_nodes(G, fac.geometry.x, fac.geometry.y)
    
    for mins in TIMES:
        # Calculate walking distance for this time
        dist = mins * METERS_PER_MIN
        
        # Find all reachable nodes within this distance
        subgraph = nx.ego_graph(G, node, radius=dist, distance="length")
        
        # Draw a shape around those nodes
        nodes_gdf = ox.graph_to_gdfs(subgraph, edges=False)
        hull = nodes_gdf.unary_union.convex_hull
        
        isochrones.append({
            "facility": fac.get("name", "Unknown"),
            "minutes": mins,
            "geometry": hull
        })

iso_gdf = gpd.GeoDataFrame(isochrones, crs=G.graph["crs"])
print(f"Created {len(iso_gdf)} isochrones")
iso_gdf

---

## Step 7: Map the isochrones

- **Green** = 5-minute walk (very accessible)
- **Yellow** = 10-minute walk
- **Orange** = 15-minute walk
- **Red dot** = facility location

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

colors = {5: "green", 10: "yellow", 15: "orange"}

# Plot isochrones (largest first so smaller ones appear on top)
for mins in reversed(TIMES):
    iso_gdf[iso_gdf["minutes"] == mins].plot(
        ax=ax,
        color=colors[mins],
        alpha=0.4,
        edgecolor="black",
        label=f"{mins} min"
    )

# Plot facility location
facilities.plot(ax=ax, color="red", markersize=100, zorder=5)

ax.legend()
ax.set_title("Walking Time Isochrones", fontsize=14)
ax.set_axis_off()
plt.show()

---

## Step 8: Export results

Save the isochrones to **data/processed/** for use in QGIS or further analysis.

**Output location:** All results are saved to data/processed/ folder.

In [ ]:
# Save isochrones to processed data folder
iso_gdf.to_file(PROCESSED / "isochrones.gpkg", driver="GPKG")

print(f"Saved to: {PROCESSED / 'isochrones.gpkg'}")
print("\nYou can open this file in QGIS to:")
print("- Overlay with population data")
print("- Calculate coverage statistics")
print("- Apply professional styling")
print(f"\nAll outputs are saved to: {PROCESSED}")

---

## Done!

You've completed network accessibility analysis:
1. Downloaded real street network data from OpenStreetMap
2. Calculated walking time isochrones
3. Visualized accessibility zones
4. Exported results for QGIS

**Try changing:**
- `LATITUDE, LONGITUDE` to analyze a different location
- `network_type` to `"drive"` or `"bike"`
- `TIMES` to different minute thresholds

**Save your work:**
- Colab: `File > Save a copy in Drive`
- Local: `Ctrl+S` or `Cmd+S`